In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import multivariate_normal
import pandas as pd

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

## 1. Monte Carlo Basics

### Goal

Estimate expectations under a distribution $p(x)$:

$$\mathbb{E}_{p}[f(x)] = \int f(x)p(x)dx \approx \frac{1}{N}\sum_{i=1}^N f(x^{(i)})$$

where $x^{(i)} \sim p(x)$

### Law of Large Numbers

As $N \to \infty$, the sample average converges to the true expectation.

### Central Limit Theorem

The error decreases as $O(1/\sqrt{N})$

In [ ]:
# Demonstrate Monte Carlo estimation
def monte_carlo_demo():
    """Estimate π using Monte Carlo"""
    
    # Sample points uniformly in [0,1] x [0,1]
    # Count how many fall inside unit circle
    # π ≈ 4 * (points inside circle) / (total points)
    
    n_samples_list = [100, 1000, 10000, 100000]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    
    for idx, n_samples in enumerate(n_samples_list):
        # Generate random points
        x = np.random.uniform(0, 1, n_samples)
        y = np.random.uniform(0, 1, n_samples)
        
        # Check if inside circle
        inside = (x**2 + y**2) <= 1
        
        # Estimate π
        pi_estimate = 4 * np.sum(inside) / n_samples
        error = abs(pi_estimate - np.pi)
        
        # Plot
        ax = axes[idx]
        # Plot subset of points for visibility
        plot_n = min(n_samples, 2000)
        ax.scatter(x[:plot_n][inside[:plot_n]], y[:plot_n][inside[:plot_n]], 
                  c='blue', s=1, alpha=0.5, label='Inside')
        ax.scatter(x[:plot_n][~inside[:plot_n]], y[:plot_n][~inside[:plot_n]], 
                  c='red', s=1, alpha=0.5, label='Outside')
        
        # Draw quarter circle
        theta = np.linspace(0, np.pi/2, 100)
        ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=2)
        
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect('equal')
        ax.set_title(f'N={n_samples}\nπ ≈ {pi_estimate:.6f}\nError: {error:.6f}')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

monte_carlo_demo()

print("Monte Carlo Estimation:")
print("- Error decreases as O(1/√N)")
print("- Works in high dimensions")
print("- Requires ability to sample from p(x)")

## 2. Metropolis-Hastings Algorithm

### Problem

We want to sample from $p(x)$ but can only evaluate $p(x)$ up to a constant:
$$p(x) = \frac{\tilde{p}(x)}{Z}$$

where $Z$ is unknown.

### Algorithm

1. Start with $x^{(0)}$
2. For $t = 1, 2, \ldots$:
   - Propose: $x' \sim q(x'|x^{(t-1)})$
   - Compute acceptance ratio: 
   $$\alpha = \min\left(1, \frac{\tilde{p}(x')q(x^{(t-1)}|x')}{\tilde{p}(x^{(t-1)})q(x'|x^{(t-1)})}\right)$$
   - Accept with probability $\alpha$:
     - If accept: $x^{(t)} = x'$
     - If reject: $x^{(t)} = x^{(t-1)}$

### Symmetric Proposal (Metropolis)

If $q(x'|x) = q(x|x')$ (e.g., Gaussian random walk):
$$\alpha = \min\left(1, \frac{\tilde{p}(x')}{\tilde{p}(x^{(t-1)})}\right)$$

In [ ]:
def metropolis_hastings(log_target, x_init, proposal_std, n_samples, burn_in=1000):
    """
    Metropolis-Hastings with Gaussian random walk proposal
    
    Parameters:
    - log_target: Log of target distribution (unnormalized)
    - x_init: Initial state
    - proposal_std: Standard deviation of Gaussian proposal
    - n_samples: Number of samples to generate
    - burn_in: Number of initial samples to discard
    """
    d = len(x_init)
    samples = np.zeros((n_samples + burn_in, d))
    samples[0] = x_init
    
    n_accepted = 0
    
    for t in range(1, n_samples + burn_in):
        # Propose new state (Gaussian random walk)
        x_current = samples[t-1]
        x_proposed = x_current + np.random.randn(d) * proposal_std
        
        # Compute acceptance ratio (in log space)
        log_alpha = log_target(x_proposed) - log_target(x_current)
        
        # Accept or reject
        if np.log(np.random.rand()) < log_alpha:
            samples[t] = x_proposed
            n_accepted += 1
        else:
            samples[t] = x_current
    
    acceptance_rate = n_accepted / (n_samples + burn_in)
    
    # Discard burn-in
    samples = samples[burn_in:]
    
    return samples, acceptance_rate

# Example: Sample from 2D Gaussian mixture
def log_mixture_gaussian(x):
    """Log probability of Gaussian mixture"""
    # Two components
    mu1 = np.array([2, 2])
    mu2 = np.array([-2, -2])
    sigma = np.eye(2) * 0.5
    
    log_p1 = multivariate_normal.logpdf(x, mu1, sigma)
    log_p2 = multivariate_normal.logpdf(x, mu2, sigma)
    
    # log(p1 + p2) = log(exp(log_p1) + exp(log_p2))
    return np.logaddexp(log_p1, log_p2) + np.log(0.5)

# Run MH with different proposal variances
x_init = np.array([0.0, 0.0])
n_samples = 5000
proposal_stds = [0.1, 0.5, 2.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, proposal_std in enumerate(proposal_stds):
    samples, accept_rate = metropolis_hastings(
        log_mixture_gaussian, x_init, proposal_std, n_samples, burn_in=1000)
    
    # Trace plot
    axes[0, idx].plot(samples[:, 0], alpha=0.7, linewidth=0.5)
    axes[0, idx].set_xlabel('Iteration')
    axes[0, idx].set_ylabel('x₁')
    axes[0, idx].set_title(f'σ={proposal_std}\nAccept: {accept_rate:.1%}')
    axes[0, idx].grid(True, alpha=0.3)
    
    # Scatter plot
    axes[1, idx].scatter(samples[:, 0], samples[:, 1], alpha=0.3, s=1)
    axes[1, idx].set_xlabel('x₁')
    axes[1, idx].set_ylabel('x₂')
    axes[1, idx].set_xlim(-5, 5)
    axes[1, idx].set_ylim(-5, 5)
    
    # True distribution contours
    x1 = np.linspace(-5, 5, 100)
    x2 = np.linspace(-5, 5, 100)
    X1, X2 = np.meshgrid(x1, x2)
    Z = np.zeros_like(X1)
    for i in range(len(x1)):
        for j in range(len(x2)):
            Z[j, i] = np.exp(log_mixture_gaussian(np.array([X1[j, i], X2[j, i]])))
    axes[1, idx].contour(X1, X2, Z, levels=10, alpha=0.5, colors='red')
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Metropolis-Hastings:")
print("- Small proposal σ: High acceptance, slow exploration")
print("- Large proposal σ: Low acceptance, can get stuck")
print("- Optimal acceptance rate: ~20-50% for high dimensions")

## 3. Gibbs Sampling

### Algorithm

For multivariate distribution $p(x_1, \ldots, x_d)$:

1. Initialize $\mathbf{x}^{(0)} = (x_1^{(0)}, \ldots, x_d^{(0)})$
2. For $t = 1, 2, \ldots$:
   - Sample $x_1^{(t)} \sim p(x_1 | x_2^{(t-1)}, \ldots, x_d^{(t-1)})$
   - Sample $x_2^{(t)} \sim p(x_2 | x_1^{(t)}, x_3^{(t-1)}, \ldots, x_d^{(t-1)})$
   - ...
   - Sample $x_d^{(t)} \sim p(x_d | x_1^{(t)}, \ldots, x_{d-1}^{(t)})$

### Key Property

- Acceptance rate is always 100%!
- Requires knowing conditional distributions
- Special case of Metropolis-Hastings

In [ ]:
def gibbs_sampling_gaussian(mu, Sigma, n_samples, burn_in=1000):
    """
    Gibbs sampling for 2D Gaussian
    
    For Gaussian, conditional distributions are also Gaussian:
    p(x₁|x₂) = N(μ₁ + Σ₁₂/Σ₂₂(x₂ - μ₂), Σ₁₁ - Σ₁₂²/Σ₂₂)
    """
    samples = np.zeros((n_samples + burn_in, 2))
    samples[0] = [0, 0]  # Initialize
    
    # Extract parameters
    mu1, mu2 = mu
    s11, s12, s22 = Sigma[0,0], Sigma[0,1], Sigma[1,1]
    
    for t in range(1, n_samples + burn_in):
        # Sample x₁ given x₂
        x2_current = samples[t-1, 1]
        mu_cond = mu1 + (s12/s22) * (x2_current - mu2)
        var_cond = s11 - s12**2/s22
        samples[t, 0] = np.random.randn() * np.sqrt(var_cond) + mu_cond
        
        # Sample x₂ given x₁
        x1_current = samples[t, 0]
        mu_cond = mu2 + (s12/s11) * (x1_current - mu1)
        var_cond = s22 - s12**2/s11
        samples[t, 1] = np.random.randn() * np.sqrt(var_cond) + mu_cond
    
    return samples[burn_in:]

# True distribution parameters
mu_true = np.array([1, -1])
Sigma_true = np.array([[1.0, 0.8],
                       [0.8, 1.0]])

# Run Gibbs sampling
gibbs_samples = gibbs_sampling_gaussian(mu_true, Sigma_true, n_samples=5000)

# Compare with direct sampling
direct_samples = np.random.multivariate_normal(mu_true, Sigma_true, 5000)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Gibbs trace
axes[0, 0].plot(gibbs_samples[:500, 0], alpha=0.7)
axes[0, 0].axhline(y=mu_true[0], color='r', linestyle='--', label='True mean')
axes[0, 0].set_ylabel('x₁')
axes[0, 0].set_title('Gibbs Trace (x₁)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(gibbs_samples[:500, 1], alpha=0.7)
axes[0, 1].axhline(y=mu_true[1], color='r', linestyle='--', label='True mean')
axes[0, 1].set_ylabel('x₂')
axes[0, 1].set_title('Gibbs Trace (x₂)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Show Gibbs path
axes[0, 2].plot(gibbs_samples[:100, 0], gibbs_samples[:100, 1], 
               'o-', alpha=0.5, markersize=3, linewidth=0.5)
axes[0, 2].plot(gibbs_samples[0, 0], gibbs_samples[0, 1], 'go', 
               markersize=10, label='Start')
axes[0, 2].set_xlabel('x₁')
axes[0, 2].set_ylabel('x₂')
axes[0, 2].set_title('Gibbs Sampling Path (first 100)')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Scatter plots
axes[1, 0].scatter(gibbs_samples[:, 0], gibbs_samples[:, 1], alpha=0.3, s=1)
axes[1, 0].set_xlabel('x₁')
axes[1, 0].set_ylabel('x₂')
axes[1, 0].set_title('Gibbs Samples')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].scatter(direct_samples[:, 0], direct_samples[:, 1], alpha=0.3, s=1)
axes[1, 1].set_xlabel('x₁')
axes[1, 1].set_ylabel('x₂')
axes[1, 1].set_title('Direct Samples')
axes[1, 1].grid(True, alpha=0.3)

# Histograms comparison
axes[1, 2].hist(gibbs_samples[:, 0], bins=50, alpha=0.5, density=True, label='Gibbs')
axes[1, 2].hist(direct_samples[:, 0], bins=50, alpha=0.5, density=True, label='Direct')
axes[1, 2].set_xlabel('x₁')
axes[1, 2].set_ylabel('Density')
axes[1, 2].set_title('Marginal Distribution')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Gibbs Sampling Results:")
print(f"\nTrue mean: {mu_true}")
print(f"Gibbs mean: {gibbs_samples.mean(axis=0)}")
print(f"Direct mean: {direct_samples.mean(axis=0)}")
print(f"\nTrue covariance:\n{Sigma_true}")
print(f"\nGibbs covariance:\n{np.cov(gibbs_samples.T)}")
print(f"\nDirect covariance:\n{np.cov(direct_samples.T)}")

## 4. Convergence Diagnostics

### How do we know when MCMC has converged?

1. **Trace plots**: Visual inspection of sample paths
2. **Running mean**: Should stabilize after convergence
3. **Autocorrelation**: Measure dependence between samples
4. **Effective sample size (ESS)**: Accounts for autocorrelation
5. **Gelman-Rubin statistic**: Compare multiple chains

In [ ]:
def compute_autocorrelation(samples, max_lag=100):
    """Compute autocorrelation function"""
    n = len(samples)
    samples_centered = samples - samples.mean()
    
    autocorr = np.zeros(max_lag + 1)
    variance = np.var(samples)
    
    for lag in range(max_lag + 1):
        if lag == 0:
            autocorr[lag] = 1.0
        else:
            autocorr[lag] = np.mean(
                samples_centered[:-lag] * samples_centered[lag:]
            ) / variance
    
    return autocorr

def effective_sample_size(samples):
    """Compute effective sample size"""
    n = len(samples)
    autocorr = compute_autocorrelation(samples, max_lag=min(n//2, 500))
    
    # Sum autocorrelations until they become negative
    # ESS = N / (1 + 2 * sum of autocorrelations)
    sum_autocorr = 0
    for rho in autocorr[1:]:
        if rho < 0:
            break
        sum_autocorr += rho
    
    ess = n / (1 + 2 * sum_autocorr)
    return ess

# Analyze Gibbs samples
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Running mean
running_mean = np.cumsum(gibbs_samples[:, 0]) / np.arange(1, len(gibbs_samples) + 1)
axes[0, 0].plot(running_mean, linewidth=2)
axes[0, 0].axhline(y=mu_true[0], color='r', linestyle='--', linewidth=2, label='True mean')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Running Mean')
axes[0, 0].set_title('Convergence of Mean')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Autocorrelation
autocorr = compute_autocorrelation(gibbs_samples[:, 0], max_lag=100)
axes[0, 1].bar(range(len(autocorr)), autocorr, alpha=0.7)
axes[0, 1].set_xlabel('Lag')
axes[0, 1].set_ylabel('Autocorrelation')
axes[0, 1].set_title('Autocorrelation Function')
axes[0, 1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0, 1].grid(True, alpha=0.3)

# Histogram with true density
axes[1, 0].hist(gibbs_samples[:, 0], bins=50, density=True, alpha=0.7, label='Samples')
x_range = np.linspace(-3, 5, 200)
# Marginal distribution
true_marginal = stats.norm.pdf(x_range, mu_true[0], np.sqrt(Sigma_true[0, 0]))
axes[1, 0].plot(x_range, true_marginal, 'r-', linewidth=2, label='True density')
axes[1, 0].set_xlabel('x₁')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Sample Distribution vs True')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# ESS over time
ess_values = []
sample_sizes = np.arange(100, len(gibbs_samples), 100)
for n in sample_sizes:
    ess = effective_sample_size(gibbs_samples[:n, 0])
    ess_values.append(ess)

axes[1, 1].plot(sample_sizes, ess_values, linewidth=2, label='ESS')
axes[1, 1].plot(sample_sizes, sample_sizes, 'r--', linewidth=2, label='Perfect (ESS=N)')
axes[1, 1].set_xlabel('Number of samples')
axes[1, 1].set_ylabel('Effective Sample Size')
axes[1, 1].set_title('Effective Sample Size Growth')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute final ESS
ess_final = effective_sample_size(gibbs_samples[:, 0])
print(f"\nConvergence Diagnostics:")
print(f"Total samples: {len(gibbs_samples)}")
print(f"Effective sample size: {ess_final:.0f}")
print(f"ESS ratio: {ess_final/len(gibbs_samples):.2%}")
print(f"\nInterpretation:")
print(f"  {len(gibbs_samples)} correlated samples ≈ {ess_final:.0f} independent samples")

## 5. Bayesian Linear Regression with MCMC

### Model

$$y = \mathbf{X}\mathbf{w} + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2\mathbf{I})$$

### Prior

$$\mathbf{w} \sim \mathcal{N}(\mathbf{0}, \lambda^{-1}\mathbf{I})$$

### Posterior (analytic)

$$p(\mathbf{w}|\mathbf{y}, \mathbf{X}) = \mathcal{N}(\mathbf{w}_N, \mathbf{S}_N)$$

We'll use MCMC to sample from this posterior and compare with analytic solution.

In [ ]:
# Generate regression data
np.random.seed(42)
n = 50
X_reg = np.random.randn(n, 1)
X_reg = np.column_stack([np.ones(n), X_reg])  # Add intercept
w_true = np.array([2.0, 3.0])
sigma_true = 1.0
y_reg = X_reg @ w_true + np.random.randn(n) * sigma_true

# Prior parameters
lambda_prior = 1.0

# Log posterior (up to constant)
def log_posterior(w):
    # Log likelihood
    residuals = y_reg - X_reg @ w
    log_lik = -0.5 * np.sum(residuals**2) / sigma_true**2
    
    # Log prior
    log_prior = -0.5 * lambda_prior * np.sum(w**2)
    
    return log_lik + log_prior

# MCMC sampling
w_init = np.zeros(2)
mcmc_samples, accept_rate = metropolis_hastings(
    log_posterior, w_init, proposal_std=0.2, n_samples=10000, burn_in=2000)

# Analytic posterior
S_N = np.linalg.inv(lambda_prior * np.eye(2) + (1/sigma_true**2) * X_reg.T @ X_reg)
w_N = (1/sigma_true**2) * S_N @ X_reg.T @ y_reg

# Sample from analytic posterior
analytic_samples = np.random.multivariate_normal(w_N, S_N, 10000)

print("Bayesian Linear Regression Results:")
print(f"\nTrue weights: {w_true}")
print(f"\nMCMC estimates:")
print(f"  Mean: {mcmc_samples.mean(axis=0)}")
print(f"  Std:  {mcmc_samples.std(axis=0)}")
print(f"\nAnalytic posterior:")
print(f"  Mean: {w_N}")
print(f"  Std:  {np.sqrt(np.diag(S_N))}")
print(f"\nMCMC acceptance rate: {accept_rate:.2%}")

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Trace plots
axes[0, 0].plot(mcmc_samples[:, 0], alpha=0.7, linewidth=0.5)
axes[0, 0].axhline(y=w_true[0], color='r', linestyle='--', label='True')
axes[0, 0].axhline(y=w_N[0], color='g', linestyle='--', label='Posterior mean')
axes[0, 0].set_ylabel('w₀ (intercept)')
axes[0, 0].set_title('MCMC Trace')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(mcmc_samples[:, 1], alpha=0.7, linewidth=0.5)
axes[0, 1].axhline(y=w_true[1], color='r', linestyle='--', label='True')
axes[0, 1].axhline(y=w_N[1], color='g', linestyle='--', label='Posterior mean')
axes[0, 1].set_ylabel('w₁ (slope)')
axes[0, 1].set_title('MCMC Trace')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Joint posterior
axes[0, 2].scatter(mcmc_samples[:, 0], mcmc_samples[:, 1], alpha=0.3, s=1, label='MCMC')
axes[0, 2].scatter(analytic_samples[:, 0], analytic_samples[:, 1], 
                  alpha=0.1, s=1, color='red', label='Analytic')
axes[0, 2].plot(w_true[0], w_true[1], 'g*', markersize=15, label='True')
axes[0, 2].set_xlabel('w₀')
axes[0, 2].set_ylabel('w₁')
axes[0, 2].set_title('Joint Posterior')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Marginal histograms
axes[1, 0].hist(mcmc_samples[:, 0], bins=50, density=True, alpha=0.7, label='MCMC')
axes[1, 0].hist(analytic_samples[:, 0], bins=50, density=True, alpha=0.5, label='Analytic')
axes[1, 0].axvline(x=w_true[0], color='r', linestyle='--', linewidth=2, label='True')
axes[1, 0].set_xlabel('w₀')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Marginal Posterior (Intercept)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(mcmc_samples[:, 1], bins=50, density=True, alpha=0.7, label='MCMC')
axes[1, 1].hist(analytic_samples[:, 1], bins=50, density=True, alpha=0.5, label='Analytic')
axes[1, 1].axvline(x=w_true[1], color='r', linestyle='--', linewidth=2, label='True')
axes[1, 1].set_xlabel('w₁')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Marginal Posterior (Slope)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Posterior predictive
x_test = np.linspace(-3, 3, 100)
X_test = np.column_stack([np.ones(len(x_test)), x_test])

# Sample predictions
y_pred_samples = X_test @ mcmc_samples[:1000].T
y_pred_mean = y_pred_samples.mean(axis=1)
y_pred_std = y_pred_samples.std(axis=1)

axes[1, 2].scatter(X_reg[:, 1], y_reg, alpha=0.6, label='Data')
axes[1, 2].plot(x_test, y_pred_mean, 'b-', linewidth=2, label='Posterior mean')
axes[1, 2].fill_between(x_test, 
                       y_pred_mean - 2*y_pred_std,
                       y_pred_mean + 2*y_pred_std,
                       alpha=0.3, label='95% credible interval')
axes[1, 2].plot(x_test, X_test @ w_true, 'r--', linewidth=2, label='True line')
axes[1, 2].set_xlabel('x')
axes[1, 2].set_ylabel('y')
axes[1, 2].set_title('Posterior Predictive')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In this notebook, we covered:

1. **Monte Carlo Basics**: Approximating expectations via sampling
2. **Metropolis-Hastings**: General-purpose MCMC algorithm
3. **Gibbs Sampling**: Coordinate-wise sampling for conjugate models
4. **Convergence Diagnostics**: ESS, autocorrelation, trace plots
5. **Bayesian Linear Regression**: MCMC for posterior inference

## Key Takeaways

- **MCMC**: Generate samples from complex distributions
- **Metropolis-Hastings**:
  - Works when you can evaluate p(x) up to constant
  - Tuning proposal variance is critical
  - Target 20-50% acceptance rate
- **Gibbs Sampling**:
  - Special case with 100% acceptance
  - Requires conditional distributions
  - Can be slow if variables highly correlated
- **Convergence**: 
  - Always use burn-in period
  - Monitor multiple diagnostics
  - Run multiple chains from different initializations
- **Effective Sample Size**: Accounts for autocorrelation

## Comparison

| Algorithm | Pros | Cons |
|-----------|------|------|
| Metropolis-Hastings | General, simple | Need to tune proposal |
| Gibbs | 100% acceptance | Need conditionals, slow mixing |
| HMC | Fast mixing, efficient | Complex, needs gradients |

## Modern MCMC

- **Hamiltonian Monte Carlo (HMC)**: Uses gradient information
- **NUTS**: No-U-Turn Sampler (used in Stan, PyMC3)
- **Variational Inference**: Faster approximate alternative

## Exercises

1. Implement Hamiltonian Monte Carlo
2. Derive the Gibbs sampler for a mixture of Gaussians
3. Implement the Gelman-Rubin convergence diagnostic
4. Use MCMC for Bayesian logistic regression
5. Compare MCMC with variational inference